# Monte Carlo Dosimetry for Lu-177 DOTATATE — Quickstart

This notebook walks through the full project end-to-end:

1. **Biokinetic model** — reference doses and the MIRD formula
2. **Baseline MC** — convergence at O(1/sqrt(n))
3. **Variance reduction** — four methods benchmarked against naive MC
4. **Bayesian MCMC** — posterior dose distribution from SPECT data
5. **Population analysis** — standard vs. individualized dosing at scale

Run cells top to bottom. All figures are saved to `../figures/`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import matplotlib.pyplot as plt

# Silence warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Imports OK")


## 1. Biokinetic Model and Reference Doses

In [ ]:
from simulation import (
    ORGAN_PARAMS, compute_reference_doses,
    KIDNEY_TOLERANCE_GY, TUMOR_MIN_GY, STANDARD_ACTIVITY_GBQ,
    time_activity_curve,
)

# Reference doses at standard 7.4 GBq
doses = compute_reference_doses()
print(f"Reference absorbed doses at {STANDARD_ACTIVITY_GBQ} GBq:")
print("-" * 50)
for organ, dose in doses.items():
    flag = "  <-- exceeds QUANTEC limit!" if organ == "Kidneys" and dose > KIDNEY_TOLERANCE_GY else ""
    print(f"  {organ:<15}: {dose:6.2f} Gy{flag}")


In [ ]:
# Plot the time-activity curve for kidneys
t = np.linspace(0, 200, 500)
p = ORGAN_PARAMS["Kidneys"]
tac = time_activity_curve(t, p.A_fast, p.lam_fast, p.A_slow, p.lam_slow)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, tac, color='#a855f7', lw=2, label='Kidney TAC (population mean)')
ax.fill_between(t, tac, alpha=0.15, color='#a855f7')
ax.set_xlabel('Time post-injection (hours)')
ax.set_ylabel('Fractional activity remaining')
ax.set_title('Lu-177 DOTATATE Kidney Time-Activity Curve')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/tac_example.png', dpi=120, bbox_inches='tight')
plt.show()
print("TAC plotted.")


## 2. Baseline Monte Carlo Convergence

In [ ]:
from simulation import convergence_experiment, naive_mc_dose
from visualization import plot_baseline_convergence

# Run convergence experiment (may take ~30 seconds)
sample_sizes = np.logspace(1, 4.5, 35).astype(int)
print("Running convergence experiment...")
conv = convergence_experiment(sample_sizes=sample_sizes, n_trials=60, seed=42)

print(f"True kidney dose: {conv['true_dose']:.4f} Gy")
print(f"At n=5000: error = {conv['errors'][-5]:.4f} Gy")


In [ ]:
fig = plot_baseline_convergence(conv, n_dist_trials=300)
plt.show()


## 3. Variance Reduction

In [ ]:
from variance_reduction import (
    antithetic_variates, control_variates,
    importance_sampling, quasi_monte_carlo,
    benchmark_vrf, ALL_METHODS,
)

# Quick demo: compare all methods at n=1000
n = 1000
true_dose = conv['true_dose']
print(f"Method comparison at n={n} (single run, not averaged):")
print("-" * 50)
for name, fn in ALL_METHODS.items():
    est, se = fn(n)
    print(f"  {name:<25}: {est:.4f} Gy  (se = {se:.4f})")
print(f"  {'True dose':<25}: {true_dose:.4f} Gy")


In [ ]:
# Full benchmark: VRF at n=500 over 150 trials
print("Running VRF benchmark (150 trials each, may take ~60s)...")
vrfs = benchmark_vrf(n=500, n_trials=150, seed=42)
print()
print("Variance Reduction Factors at n=500:")
print("-" * 45)
for name, vrf in vrfs.items():
    bar = '#' * min(int(vrf / 5), 30)
    print(f"  {name:<25}: {vrf:7.1f}x  {bar}")


In [ ]:
# Build convergence data for all methods
sample_sizes_p2 = np.logspace(1, 4.2, 25).astype(int)
results = {name: {'errors': [], 'stds': []} for name in ALL_METHODS}

print("Building convergence curves (this takes a minute)...")
for sz in sample_sizes_p2:
    for name, fn in ALL_METHODS.items():
        trials = [fn(sz)[0] for _ in range(30)]
        results[name]['errors'].append(abs(np.mean(trials) - true_dose))
        results[name]['stds'].append(np.std(trials))
    if sz in sample_sizes_p2[::5]:
        print(f"  n={sz} done")

for name in results:
    results[name]['errors'] = np.array(results[name]['errors'])

from visualization import plot_variance_reduction
fig = plot_variance_reduction(results, vrfs, n_vrf=500,
                               sample_sizes=sample_sizes_p2)
plt.show()


## 4. Bayesian MCMC — Dose Uncertainty from SPECT Data

In [ ]:
from mcmc import simulate_spect_measurements, metropolis_hastings, posterior_summary, SCAN_TIMES

# Simulate SPECT measurements for one patient
observed, true_vals = simulate_spect_measurements(seed=17)
print("Simulated SPECT measurements:")
print(f"  Timepoints: {SCAN_TIMES} hours")
print(f"  True TAC:   {np.round(true_vals, 4)}")
print(f"  Observed:   {np.round(observed, 4)}")
print(f"  Noise CV:   8%")


In [ ]:
# Run MCMC (takes ~10-20 seconds)
print("Running Metropolis-Hastings MCMC (20,000 iterations)...")
mcmc_result = metropolis_hastings(observed, seed=17)
print(f"Acceptance rate: {mcmc_result.acceptance_rate:.2%}")

summary = posterior_summary(mcmc_result)
print()
print("Posterior Summary:")
print(f"  A_slow:    mean={summary['A_slow']['mean']:.4f}  "
      f"true={summary['A_slow']['true']:.4f}  "
      f"95% CI={[round(x,4) for x in summary['A_slow']['ci95']]}")
print(f"  lam_slow:  mean={summary['lam_slow']['mean']:.5f}  "
      f"true={summary['lam_slow']['true']:.5f}  "
      f"95% CI={[round(x,5) for x in summary['lam_slow']['ci95']]}")
print(f"  Dose (Gy): mean={summary['kidney_dose_Gy']['mean']:.2f}  "
      f"true={summary['kidney_dose_Gy']['true']:.2f}  "
      f"95% CI={[round(x,2) for x in summary['kidney_dose_Gy']['ci95']]}")
print(f"  P(dose > 23 Gy QUANTEC limit) = {summary['kidney_dose_Gy']['p_exceed_limit']:.1%}")


In [ ]:
from visualization import plot_mcmc_results
fig = plot_mcmc_results(mcmc_result)
plt.show()


### Interpreting the MCMC result

The posterior predictive dose spans 25-48 Gy with 99.8% probability of exceeding
the 23 Gy QUANTEC limit. This is not a failure of the method — it reflects genuine
uncertainty from having only 4 SPECT timepoints. The slow-component clearance rate
`lambda_slow` is hard to pin down without a late imaging scan (168-240 h).

This makes the case for acquiring a late timepoint, which is an active debate
in the EANM dosimetry guidelines.


## 5. Population-Level Safety Analysis

In [ ]:
from population_analysis import compare_dosing_strategies, print_comparison

# Simulate 8,000 patients (takes a few seconds)
print("Simulating 8,000 patients under both dosing strategies...")
comparison = compare_dosing_strategies(n=8_000, seed=42)
print()
print_comparison(comparison)


In [ ]:
from visualization import plot_population_analysis
fig = plot_population_analysis(comparison)
plt.show()


### Key takeaway

| Metric | Standard 7.4 GBq | Individualized MC |
|---|---|---|
| P(kidney toxicity) | 65.8% | 4.0% |
| P(under-treatment) | 1.8% | 15.8% |
| P(optimal outcome) | 32.5% | 80.2% |

Individualized MC dosing reduces kidney toxicity risk **16-fold**.
The trade-off is a modest increase in under-treatment for patients whose
tumors clear the drug so fast that hitting tumoricidal dose would exceed
the kidney limit. That's a biological constraint, not a modeling failure.


## 6. Connections to ML

In [ ]:
# The control variate here is mathematically identical to a REINFORCE baseline.
# Let's verify: compute variance with and without the control variate.

from simulation import sample_patient_doses
from variance_reduction import control_variates

n = 2000
raw_doses = sample_patient_doses(n, seed=42)
cv_doses, _ = control_variates(n)

raw_var = np.var(raw_doses)
cv_samples = np.array([control_variates(n)[0] for _ in range(200)])
cv_var = np.var(cv_samples)
naive_samples = np.array([sample_patient_doses(n, seed=i).mean() for i in range(200)])
naive_var = np.var(naive_samples)

print("Variance comparison (200 trials each, n=2000):")
print(f"  Naive MC variance:          {naive_var:.6f}")
print(f"  Control variate variance:   {cv_var:.6f}")
print(f"  VRF: {naive_var / cv_var:.2f}x")
print()
print("In RL (REINFORCE), subtracting a value function baseline from the")
print("return does exactly this: reduces gradient variance without bias.")


---

## Summary

This project shows that:

1. Standard Lu-177 DOTATATE dosing is statistically unsafe for the majority
   of patients with typical inter-patient variability.

2. Variance reduction techniques make real-time patient-specific dosimetry
   computationally feasible — Quasi-MC gives 167x speedup over naive MC.

3. Bayesian MCMC turns sparse SPECT data into a full dose uncertainty
   distribution, enabling proper safety exceedance probability statements.

4. MC-informed individualized dosing reduces kidney toxicity risk 16-fold
   while maintaining therapeutic efficacy in most patients.

The math here — variance reduction, importance sampling, MCMC — is the
same math that powers variational autoencoders, policy gradient RL,
and probabilistic programming systems like Stan and NumPyro.
